# Support Vector Regression (SVR) and Tree-Based Regression

## 📚 Learning Objectives

By completing this notebook, you will:
- Build SVR models with different kernels (linear, RBF, polynomial)
- Implement decision tree and random forest regression
- Compare tree-based models with SVR
- Understand when to use each approach

## 🔗 Where this fits

**Builds on:** Course 04 — Unit 1, lesson 06 "Ridge and Lasso Regression" — linear models with a penalty; SVR and trees drop the straight-line assumption altogether.

**Used later in:** Course 04 — Unit 3, where the same kernel and tree machinery is turned into classifiers.

---

This notebook covers practical activities from **Course 04, Unit 1**:
- Building SVR models with different kernels
- Implementing decision tree and random forest regression


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- **Real data**: `../../datasets/raw/montgomery_911_calls.csv` — 663,522 emergency calls
  dispatched in Montgomery County, PA, with a real timestamp on every one
- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# CELL: Import the three non-linear regressors compared in this lesson.
# WHY: SVR, a single decision tree, and a random forest attack the same
# curve-fitting problem in very different ways - we want to compare them fairly.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# CELL: Build a REAL non-linear regression problem from the 911 dispatch log.
# WHY: emergency-call volume genuinely rises and falls over the day — quiet before dawn,
# peaking in the late afternoon. That curve is not something we invented; it is what the
# county actually recorded, and it is exactly the kind of shape a straight line cannot fit.
calls = pd.read_csv("../../datasets/raw/montgomery_911_calls.csv", usecols=["timeStamp"])
timestamps = pd.to_datetime(calls["timeStamp"])

# Count how many calls arrived in each clock hour of each day in the log.
hourly = timestamps.dt.floor("h").value_counts().sort_index()
volume = pd.DataFrame({"timestamp": hourly.index, "calls": hourly.values})
volume["hour_of_day"] = volume["timestamp"].dt.hour.astype(float)

print(f"Dispatch log: {len(calls):,} real emergency calls")
print(f"Aggregated into {len(volume):,} hourly buckets")

# Classroom-size the problem: 600 randomly chosen hours, random_state=42.
sample = volume.sample(n=600, random_state=42)
X = sample[["hour_of_day"]].to_numpy()          # feature: hour of day, 0-23
y = sample["calls"].to_numpy().astype(float)    # target: calls dispatched in that hour

# Split into train/test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Target range:     [{y.min():.0f}, {y.max():.0f}] calls per hour")

# Show the real shape we are asking the models to learn.
profile = volume.groupby("hour_of_day")["calls"].mean()
print("\nAverage calls per hour across the whole log (the real curve):")
for h in [0, 4, 8, 12, 16, 17, 20, 23]:
    bar = "#" * int(round(profile[h]))
    print(f"  {int(h):02d}:00  {profile[h]:5.1f}  {bar}")
print(f"\nQuietest hour: {int(profile.idxmin()):02d}:00 ({profile.min():.1f} calls)   "
      f"Busiest hour: {int(profile.idxmax()):02d}:00 ({profile.max():.1f} calls)")

Dispatch log: 663,522 real emergency calls
Aggregated into 40,546 hourly buckets

Training samples: 480
Test samples:     120
Target range:     [1, 129] calls per hour

Average calls per hour across the whole log (the real curve):
  00:00    8.2  ########
  04:00    5.5  ######
  08:00   19.7  ####################
  12:00   23.7  ########################
  16:00   25.3  #########################
  17:00   26.1  ##########################
  20:00   16.6  #################
  23:00   10.0  ##########

Quietest hour: 04:00 (5.5 calls)   Busiest hour: 17:00 (26.1 calls)


## Part 1: Support Vector Regression with Different Kernels

**New model - the 2-minute intuition (full theory in Unit 3, Example 3):**
Linear regression fits a line by penalizing *every* error. **SVR (Support Vector Regression)** instead fits a *tube* around the data: points inside the tube cost nothing, and the fit is shaped only by the points on or outside the tube (the *support vectors*). A **kernel** decides what shape the tube can take:

- `linear` - a straight tube (like a straight line fit)
- `poly` - a curved, polynomial-shaped tube
- `rbf` - a flexible tube that can follow almost any smooth curve

That is all you need for this notebook - treat the kernels as three levels of flexibility and compare their errors.


In [3]:
# SVR with different kernels
kernels = ['linear', 'rbf', 'poly']
svr_models = {}

print("SVR performance by kernel:")
print("-" * 44)
# Train one SVR per kernel and score each on the same held-out test points.
for kernel in kernels:
    svr = SVR(kernel=kernel, C=1.0)
    svr.fit(X_train, y_train)
    pred = svr.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    svr_models[kernel] = svr
    print(f"  {kernel:<8} kernel:  MSE = {mse:.4f}   R² = {r2:.4f}")

print()
print("The real daily call curve rises to a late-afternoon peak and falls again, so a")
print("straight line cannot follow it. Compare the linear kernel against RBF above:")
print("the linear kernel is barely better than predicting the average, while RBF")
print("tracks the real shape.")
print()
print("Note also the ceiling on R\u00b2. Hour of day is only PART of the story: a hospital")
print("does not know in advance which Tuesday will be busy. The unexplained variance is")
print("real day-to-day randomness in when people call for help, not a bug in the model.")

SVR performance by kernel:
--------------------------------------------
  linear   kernel:  MSE = 81.4593   R² = 0.0606
  rbf      kernel:  MSE = 45.5961   R² = 0.4742
  poly     kernel:  MSE = 90.6358   R² = -0.0452

The real daily call curve rises to a late-afternoon peak and falls again, so a
straight line cannot follow it. Compare the linear kernel against RBF above:
the linear kernel is barely better than predicting the average, while RBF
tracks the real shape.

Note also the ceiling on R². Hour of day is only PART of the story: a hospital
does not know in advance which Tuesday will be busy. The unexplained variance is
real day-to-day randomness in when people call for help, not a bug in the model.


## Part 2: Decision Tree and Random Forest Regression

**New models - the 2-minute intuition (full theory in Unit 3, Examples 2 and 5):**
A **decision tree** predicts by recursively splitting the input range with simple questions ("is x < 2.3?" → yes/no) and predicting the *average y* of the training points in each final region - the result is a staircase-shaped fit. A **random forest** trains many trees, each on a random sample of the data, and *averages* their predictions - the averaging smooths the staircase and reduces overfitting. Here, just compare their test errors; how splits are chosen and why averaging works comes in Unit 3.


In [4]:
# Decision Tree Regressor
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_mse = mean_squared_error(y_test, dt_pred)
dt_r2 = r2_score(y_test, dt_pred)

# Random Forest Regressor (ensemble of trees)
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

# Compare all models on the same test set
best_svr_kernel = min(svr_models, key=lambda k: mean_squared_error(y_test, svr_models[k].predict(X_test)))
best_svr_mse = mean_squared_error(y_test, svr_models[best_svr_kernel].predict(X_test))

print("Model comparison (test MSE, lower is better):")
print("-" * 44)
print(f"  SVR (best: {best_svr_kernel}):  MSE = {best_svr_mse:.4f}")
print(f"  Decision Tree:      MSE = {dt_mse:.4f}   R² = {dt_r2:.4f}")
print(f"  Random Forest:      MSE = {rf_mse:.4f}   R² = {rf_r2:.4f}")
print()
print("Random Forest averages many trees, so it is usually more")
print("robust than a single decision tree on noisy data.")
print()
print("Honest note on this dataset: with a single input feature that takes only 24")
print("distinct values, the tree can already place a split at every hour, so averaging")
print("100 of them has almost nothing left to smooth. The forest's advantage shows up")
print("when there are many features and the trees genuinely disagree \u2014 see Unit 3.")

Model comparison (test MSE, lower is better):
--------------------------------------------
  SVR (best: rbf):  MSE = 45.5961
  Decision Tree:      MSE = 45.5894   R² = 0.4743
  Random Forest:      MSE = 47.0695   R² = 0.4572

Random Forest averages many trees, so it is usually more
robust than a single decision tree on noisy data.

Honest note on this dataset: with a single input feature that takes only 24
distinct values, the tree can already place a split at every hour, so averaging
100 of them has almost nothing left to smooth. The forest's advantage shows up
when there are many features and the trees genuinely disagree — see Unit 3.


## Summary

### Key Concepts:
1. **SVR Kernels**: Linear (simple), RBF (non-linear), Polynomial (curved)
2. **Decision Tree**: Simple, interpretable, prone to overfitting
3. **Random Forest**: Ensemble of trees, more robust, less overfitting
4. **When to use**: SVR for non-linear, Tree-based for interpretability
5. **Real data has a ceiling**: on the real 911 dispatch log, hour of day explains only
   part of the variation in call volume. A non-linear model closes the gap that a linear
   model cannot — but no model can predict the part that is genuinely unpredictable.

**Reference:** Course 04, Unit 1: "Building SVR models with different kernels" and "Implementing decision tree and random forest regression"


## 📚 References

1. Drucker, H., Burges, C. J. C., Kaufman, L., Smola, A., & Vapnik, V. (1997). *Support Vector Regression Machines*. Advances in Neural Information Processing Systems 9.
2. Breiman, L., Friedman, J., Olshen, R., & Stone, C. (1984). *Classification and Regression Trees*. Wadsworth.
3. Smola, A. J., & Schölkopf, B. (2004). *A Tutorial on Support Vector Regression*. Statistics and Computing, 14, 199–222.
4. Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). *Why Do Tree-Based Models Still Outperform Deep Learning on Tabular Data?* NeurIPS 2022 Datasets and Benchmarks. <https://arxiv.org/abs/2207.08815>
